In [1]:
# Import libraries, configure display settings, and initialize BigQuery client.
import warnings

import pandas as pd
from google.cloud import bigquery
import pybaseball

PROJECT_ID = "baseball-analytics-portfolio"

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:.3f}".format)

warnings.filterwarnings("ignore", module="pybaseball")

client = bigquery.Client(project=PROJECT_ID)
print("Environment ready")


Environment ready


# Baseball Analytics - Data Profiling Notebook
Quick reference for exploring raw, staging, and mart tables in BigQuery.  
All queries run against baseball-analytics-portfolio.


In [2]:
# Define reusable helpers for running SQL and previewing BigQuery tables.
def run_query(sql: str) -> pd.DataFrame:
    """Run a BigQuery SQL query and return results as a DataFrame."""
    return client.query(sql).result().to_dataframe()


def preview_table(dataset: str, table: str, limit: int = 10) -> pd.DataFrame:
    """Preview first N rows of any BigQuery table."""
    sql = f"SELECT * FROM `{PROJECT_ID}.{dataset}.{table}` LIMIT {int(limit)}"
    return run_query(sql)


## Raw Layer - statcast_pitches


In [3]:
# Summarize raw Statcast table coverage and pitch-type distribution.
raw_summary_sql = f"""
SELECT
  COUNT(*) AS total_rows,
  COUNT(DISTINCT DATE(TIMESTAMP_SECONDS(SAFE_CAST(game_date / 1000000000 AS INT64)))) AS distinct_game_dates,
  MIN(DATE(TIMESTAMP_SECONDS(SAFE_CAST(game_date / 1000000000 AS INT64)))) AS min_game_date,
  MAX(DATE(TIMESTAMP_SECONDS(SAFE_CAST(game_date / 1000000000 AS INT64)))) AS max_game_date,
  COUNT(DISTINCT pitcher) AS distinct_pitchers,
  COUNT(DISTINCT batter) AS distinct_batters
FROM `{PROJECT_ID}.raw.statcast_pitches`
"""

raw_pitch_type_sql = f"""
SELECT
  pitch_type,
  COUNT(*) AS pitch_count
FROM `{PROJECT_ID}.raw.statcast_pitches`
GROUP BY pitch_type
ORDER BY pitch_count DESC
"""

raw_summary_df = run_query(raw_summary_sql)
raw_pitch_type_df = run_query(raw_pitch_type_sql)

print("Raw table summary:")
display(raw_summary_df)
print("Distinct pitch types with counts:")
display(raw_pitch_type_df)


Raw table summary:


,total_rows,distinct_game_dates,min_game_date,max_game_date,distinct_pitchers,distinct_batters
0,31191,8,2026-05-19,2026-05-26,424,407


Distinct pitch types with counts:


,pitch_type,pitch_count
0,FF,9813
1,SI,4865
2,SL,4386
3,CH,3401
4,ST,2454
5,FC,2414
6,CU,1943
7,FS,1030
8,KC,612
9,SV,151


In [4]:
# Preview a small sample of rows from the raw statcast table.
preview_table("raw", "statcast_pitches", limit=5)


,pitch_type,game_date,release_speed,release_pos_x,release_pos_z,player_name,batter,pitcher,events,description,spin_dir,spin_rate_deprecated,break_angle_deprecated,break_length_deprecated,zone,des,game_type,stand,p_throws,home_team,away_team,type,hit_location,bb_type,balls,strikes,game_year,pfx_x,pfx_z,plate_x,plate_z,on_3b,on_2b,on_1b,outs_when_up,inning,inning_topbot,hc_x,hc_y,tfs_deprecated,tfs_zulu_deprecated,umpire,sv_id,vx0,vy0,vz0,ax,ay,az,sz_top,sz_bot,hit_distance_sc,launch_speed,launch_angle,effective_speed,release_spin_rate,release_extension,game_pk,fielder_2,fielder_3,fielder_4,fielder_5,fielder_6,fielder_7,fielder_8,fielder_9,release_pos_y,estimated_ba_using_speedangle,estimated_woba_using_speedangle,woba_value,woba_denom,babip_value,iso_value,launch_speed_angle,at_bat_number,pitch_number,pitch_name,home_score,away_score,bat_score,fld_score,post_away_score,post_home_score,post_bat_score,post_fld_score,if_fielding_alignment,of_fielding_alignment,spin_axis,delta_home_win_exp,delta_run_exp,bat_speed,swing_length,estimated_slg_using_speedangle,delta_pitcher_run_exp,hyper_speed,home_score_diff,bat_score_diff,home_win_exp,bat_win_exp,age_pit_legacy,age_bat_legacy,age_pit,age_bat,n_thruorder_pitcher,n_priorpa_thisgame_player_at_bat,pitcher_days_since_prev_game,batter_days_since_prev_game,pitcher_days_until_next_game,batter_days_until_next_game,api_break_z_with_gravity,api_break_x_arm,api_break_x_batter_in,arm_angle,attack_angle,attack_direction,swing_path_tilt,intercept_ball_minus_batter_pos_x_inches,intercept_ball_minus_batter_pos_y_inches
0,FS,1779667200000000000,83.800,-1.380,5.330,"Miller, Bryce",680869,682243,strikeout,swinging_strike,<NA>,<NA>,<NA>,<NA>,13,Zack Gelof strikes out swinging.,R,R,R,ATH,SEA,S,2,None,1,2,2026,-1.170,-0.040,-0.176,0.731,<NA>,<NA>,<NA>,2,9,Bot,NaN,NaN,<NA>,<NA>,<NA>,<NA>,5.109,-121.857,-3.879,-12.640,24.743,-31.963,3.292,1.661,<NA>,NaN,<NA>,84.000,1150,6.500,825005,640902,647304,702284,806068,641487,668227,645302,670042,53.960,NaN,0.000,0.000,1,0,0,<NA>,76,4,Split-Finger,2,9,2,9,9,2,2,9,Infield shade,Standard,222,-0.001,-0.164,69.100,8.400,NaN,0.164,NaN,-7,-7,0.001,0.001,27,26,28,27,2,3,6,1,<NA>,<NA>,3.300,1.170,1.170,42.300,23.697,-18.386,39.808,35.717,39.519
1,FS,1779667200000000000,85.000,-1.380,5.330,"Miller, Bryce",680869,682243,None,swinging_strike,<NA>,<NA>,<NA>,<NA>,8,None,R,R,R,ATH,SEA,S,<NA>,None,1,1,2026,-0.860,0.140,-0.124,2.051,<NA>,<NA>,<NA>,2,9,Bot,NaN,NaN,<NA>,<NA>,<NA>,<NA>,4.663,-123.784,-1.487,-9.791,23.664,-30.601,3.292,1.661,<NA>,NaN,<NA>,85.300,1148,6.300,825005,640902,647304,702284,806068,641487,668227,645302,670042,54.160,NaN,NaN,NaN,<NA>,<NA>,<NA>,<NA>,76,3,Split-Finger,2,9,2,9,9,2,2,9,Infield shade,Standard,215,0.000,-0.068,71.300,8.100,NaN,0.068,NaN,-7,-7,0.001,0.001,27,26,28,27,2,3,6,1,<NA>,<NA>,2.990,0.860,0.860,45.400,26.849,-25.426,31.656,36.558,39.143
2,FF,1779667200000000000,95.900,-0.910,5.650,"Miller, Bryce",680869,682243,None,called_strike,<NA>,<NA>,<NA>,<NA>,4,None,R,R,R,ATH,SEA,S,<NA>,None,1,0,2026,-0.230,1.490,-0.327,2.175,<NA>,<NA>,<NA>,2,9,Bot,NaN,NaN,<NA>,<NA>,<NA>,<NA>,2.042,-139.586,-6.818,-3.422,29.857,-11.314,3.292,1.661,<NA>,NaN,<NA>,96.000,2497,6.200,825005,640902,647304,702284,806068,641487,668227,645302,670042,54.290,NaN,NaN,NaN,<NA>,<NA>,<NA>,<NA>,76,2,4-Seam Fastball,2,9,2,9,9,2,2,9,Infield shade,Standard,204,0.000,-0.050,NaN,NaN,NaN,0.050,NaN,-7,-7,0.001,0.001,27,26,28,27,2,3,6,1,<NA>,<NA>,0.970,0.230,0.230,48.200,NaN,NaN,NaN,NaN,NaN
3,SL,1779667200000000000,91.100,-1.170,5.630,"Miller, Bryce",680869,682243,None,ball,<NA>,<NA>,<NA>,<NA>,12,None,R,R,R,ATH,SEA,B,<NA>,None,0,0,2026,0.570,0.840,0.650,3.979,<NA>,<NA>,<NA>,2,9,Bot,NaN,NaN,<NA>,<NA>,<NA>,<NA>,3.268,-132.713,0.017,6.158,25.903,-22.368,3.292,1.661,<NA>,NaN,<NA>,91.400,2557,6.200,825005,640902,647304,702284,806068,641487,668227,645302,670042,54.300,NaN,NaN,NaN,<NA>,<NA>,<NA>,<NA>,76,1,Slider,2,9,2,9,9,2,2,9,Standard,Standard,186,0.000,0.040,NaN,NaN,NaN,-0.040,NaN,-7,-7,0.001,0.001,

## Staging Layer - stg_statcast_pitches


In [5]:
# Validate staging row counts, null checks, date range, and outcome distribution.
stg_summary_sql = f"""
SELECT
  COUNT(*) AS total_rows,
  COUNTIF(pitch_type IS NULL) AS null_pitch_type_rows,
  COUNTIF(game_date IS NULL) AS null_game_date_rows,
  MIN(game_date) AS min_game_date,
  MAX(game_date) AS max_game_date
FROM `{PROJECT_ID}.baseball_analytics.stg_statcast_pitches`
"""

stg_outcome_sql = f"""
WITH base AS (
  SELECT pitch_outcome_category
  FROM `{PROJECT_ID}.baseball_analytics.stg_statcast_pitches`
), totals AS (
  SELECT COUNT(*) AS total_rows FROM base
)
SELECT
  b.pitch_outcome_category,
  COUNT(*) AS outcome_count,
  SAFE_DIVIDE(COUNT(*), t.total_rows) AS outcome_pct
FROM base b
CROSS JOIN totals t
GROUP BY b.pitch_outcome_category, t.total_rows
ORDER BY outcome_count DESC
"""

stg_summary_df = run_query(stg_summary_sql)
stg_outcome_df = run_query(stg_outcome_sql)

print("Staging table checks:")
display(stg_summary_df)
print("pitch_outcome_category distribution (counts and percentages):")
display(stg_outcome_df)


Staging table checks:


,total_rows,null_pitch_type_rows,null_game_date_rows,min_game_date,max_game_date
0,31113,0,0,2026-05-19,2026-05-26


pitch_outcome_category distribution (counts and percentages):


,pitch_outcome_category,outcome_count,outcome_pct
0,S,14541,0.467
1,B,11110,0.357
2,X,5462,0.176


In [6]:
# Profile key numeric columns with non-null count, average, min, max, and stddev.
stg_numeric_profile_sql = f"""
SELECT 'pitch_velocity_mph' AS column_name, COUNT(pitch_velocity_mph) AS non_null_count, AVG(pitch_velocity_mph) AS avg_value, MIN(pitch_velocity_mph) AS min_value, MAX(pitch_velocity_mph) AS max_value, STDDEV(pitch_velocity_mph) AS stddev_value
FROM `{PROJECT_ID}.baseball_analytics.stg_statcast_pitches`
UNION ALL
SELECT 'spin_rate_rpm', COUNT(spin_rate_rpm), AVG(spin_rate_rpm), MIN(spin_rate_rpm), MAX(spin_rate_rpm), STDDEV(spin_rate_rpm)
FROM `{PROJECT_ID}.baseball_analytics.stg_statcast_pitches`
UNION ALL
SELECT 'exit_velocity_mph', COUNT(exit_velocity_mph), AVG(exit_velocity_mph), MIN(exit_velocity_mph), MAX(exit_velocity_mph), STDDEV(exit_velocity_mph)
FROM `{PROJECT_ID}.baseball_analytics.stg_statcast_pitches`
UNION ALL
SELECT 'launch_angle_deg', COUNT(launch_angle_deg), AVG(launch_angle_deg), MIN(launch_angle_deg), MAX(launch_angle_deg), STDDEV(launch_angle_deg)
FROM `{PROJECT_ID}.baseball_analytics.stg_statcast_pitches`
UNION ALL
SELECT 'xba', COUNT(xba), AVG(xba), MIN(xba), MAX(xba), STDDEV(xba)
FROM `{PROJECT_ID}.baseball_analytics.stg_statcast_pitches`
UNION ALL
SELECT 'xwoba', COUNT(xwoba), AVG(xwoba), MIN(xwoba), MAX(xwoba), STDDEV(xwoba)
FROM `{PROJECT_ID}.baseball_analytics.stg_statcast_pitches`
UNION ALL
SELECT 'bat_speed', COUNT(bat_speed), AVG(bat_speed), MIN(bat_speed), MAX(bat_speed), STDDEV(bat_speed)
FROM `{PROJECT_ID}.baseball_analytics.stg_statcast_pitches`
UNION ALL
SELECT 'swing_length', COUNT(swing_length), AVG(swing_length), MIN(swing_length), MAX(swing_length), STDDEV(swing_length)
FROM `{PROJECT_ID}.baseball_analytics.stg_statcast_pitches`
ORDER BY column_name
"""

stg_numeric_profile_df = run_query(stg_numeric_profile_sql)
stg_numeric_profile_df


,column_name,non_null_count,avg_value,min_value,max_value,stddev_value
0,bat_speed,14560,69.935,1.200,87.900,9.465
1,exit_velocity_mph,10339,82.644,7.400,116.900,15.095
2,launch_angle_deg,10351,18.959,-89.000,90.000,32.742
3,pitch_velocity_mph,31113,89.557,33.400,103.400,6.212
4,spin_rate_rpm,31109,2268.556,106.000,3599.000,367.307
5,swing_length,14560,7.209,0.600,10.000,1.036
6,xba,5349,0.319,0.001,1.000,0.295
7,xwoba,7999,0.309,0.000,2.044,0.371


In [7]:
# Analyze pitch-type distribution with average velocity, spin, and xBA.
pitch_type_distribution_sql = f"""
SELECT
  pitch_type,
  pitch_name,
  COUNT(*) AS pitch_count,
  AVG(pitch_velocity_mph) AS avg_pitch_velocity_mph,
  AVG(spin_rate_rpm) AS avg_spin_rate_rpm,
  AVG(xba) AS avg_xba
FROM `{PROJECT_ID}.baseball_analytics.stg_statcast_pitches`
GROUP BY pitch_type, pitch_name
ORDER BY pitch_count DESC
"""

pitch_type_distribution_df = run_query(pitch_type_distribution_sql)
pitch_type_distribution_df


,pitch_type,pitch_name,pitch_count,avg_pitch_velocity_mph,avg_spin_rate_rpm,avg_xba
0,FF,4-Seam Fastball,9813,94.683,2309.340,0.313
1,SI,Sinker,4865,94.065,2194.467,0.331
2,SL,Slider,4386,86.286,2435.817,0.313
3,CH,Changeup,3401,85.811,1737.936,0.324
4,ST,Sweeper,2454,83.015,2610.309,0.324
5,FC,Cutter,2414,89.560,2395.080,0.325
6,CU,Curveball,1943,79.608,2608.858,0.304
7,FS,Split-Finger,1030,86.652,1373.208,0.292
8,KC,Knuckle Curve,612,82.636,2512.018,0.341
9,SV,Slurve,151,81.813,2474.722,0.325


## Mart Layer - Batter Game Stats


In [8]:
# Summarize batter mart volume and key rate metrics across all rows.
mart_batter_overview_sql = f"""
SELECT
  COUNT(*) AS total_rows,
  COUNT(DISTINCT batter_id) AS distinct_batters,
  COUNT(DISTINCT game_id) AS distinct_games,
  AVG(hard_hit_rate) AS avg_hard_hit_rate,
  AVG(barrel_rate) AS avg_barrel_rate,
  AVG(avg_xwoba) AS avg_avg_xwoba,
  AVG(avg_bat_speed) AS avg_avg_bat_speed
FROM `{PROJECT_ID}.baseball_analytics.mart_batter_game_stats`
"""

mart_batter_overview_df = run_query(mart_batter_overview_sql)
print("Batter mart overview:")
display(mart_batter_overview_df)
print("Note: avg_bat_speed is expected to be mostly null for older seasons.")


Batter mart overview:


,total_rows,distinct_batters,distinct_games,avg_hard_hit_rate,avg_barrel_rate,avg_avg_xwoba,avg_avg_bat_speed
0,2217,407,108,0.248,0.044,0.296,69.862


Note: avg_bat_speed is expected to be mostly null for older seasons.


In [ ]:
# Find top batters by average exit velocity with at least 5 total batted balls.
top_batters_sql = f"""
SELECT
  batter_id,
  COUNT(*) AS games,
  SUM(batted_balls) AS total_batted_balls,
  AVG(avg_exit_velocity_mph) AS avg_exit_velocity_mph,
  AVG(avg_xwoba) AS avg_xwoba,
  SAFE_DIVIDE(SUM(hard_hit_count), SUM(batted_balls)) AS avg_hard_hit_rate,
  SUM(home_runs) AS total_home_runs
FROM `{PROJECT_ID}.baseball_analytics.mart_batter_game_stats`
GROUP BY batter_id
HAVING SUM(batted_balls) >= 5
ORDER BY avg_exit_velocity_mph DESC
LIMIT 20
"""

top_batters_df = run_query(top_batters_sql)
top_batters_df


## Mart Layer - Pitcher Game Stats


In [10]:
# Summarize pitcher mart volume and core performance metrics.
mart_pitcher_overview_sql = f"""
SELECT
  COUNT(*) AS total_rows,
  COUNT(DISTINCT pitcher_id) AS distinct_pitchers,
  COUNT(DISTINCT game_id) AS distinct_games,
  AVG(whiff_rate) AS avg_whiff_rate,
  AVG(strike_pct) AS avg_strike_pct,
  AVG(avg_pitch_velocity_mph) AS avg_avg_pitch_velocity_mph
FROM `{PROJECT_ID}.baseball_analytics.mart_pitcher_game_stats`
"""

mart_pitcher_overview_df = run_query(mart_pitcher_overview_sql)
print("Pitcher mart overview:")
display(mart_pitcher_overview_df)


Pitcher mart overview:


,total_rows,distinct_pitchers,distinct_games,avg_whiff_rate,avg_strike_pct,avg_avg_pitch_velocity_mph
0,883,424,108,0.238,0.468,89.618


In [18]:
# Find top pitchers by whiff rate with at least 50 total pitches.
top_pitchers_sql = f"""
WITH agg AS (
  SELECT
    pitcher_id,
    pitcher_name,
    COUNT(*) AS games,
    SUM(total_pitches) AS total_pitches,
    AVG(avg_pitch_velocity_mph) AS avg_velocity,
    SAFE_DIVIDE(SUM(whiffs), SUM(swings)) AS avg_whiff_rate,
    SAFE_DIVIDE(SUM(strikes), SUM(total_pitches)) AS avg_strike_pct,
    SAFE_DIVIDE(SUM(hard_hit_allowed_count), SUM(batted_balls_allowed)) AS avg_hard_hit_allowed_rate
  FROM `{PROJECT_ID}.baseball_analytics.mart_pitcher_game_stats`
  GROUP BY pitcher_id, pitcher_name
)
SELECT *
FROM agg
WHERE total_pitches >= 50
ORDER BY avg_whiff_rate DESC
LIMIT 20
"""

top_pitchers_df = run_query(top_pitchers_sql)
top_pitchers_df

,pitcher_id,pitcher_name,games,total_pitches,avg_velocity,avg_whiff_rate,avg_strike_pct,avg_hard_hit_allowed_rate
0,680755,"Fisher, Braydon",4,57,85.197,0.458,0.561,0.250
1,650911,"Sánchez, Cristopher",1,96,89.606,0.447,0.417,0.480
2,689147,"Kerkering, Orion",4,65,91.916,0.444,0.554,0.154
3,656945,"Scott, Tanner",3,56,92.862,0.429,0.571,0.167
4,684049,"Bidois, Brandan",3,64,92.513,0.424,0.531,0.188
5,656550,"Holmes, Grant",1,87,87.072,0.419,0.552,0.261
6,670970,"Morejon, Adrian",4,56,95.999,0.405,0.571,0.150
7,664126,"Fairbanks, Pete",3,56,93.067,0.400,0.589,0.250
8,669270,"Kuhnel, Joel",4,62,91.654,0.400,0.468,0.267
9,696147,"Bachman, Sam",3,52,92.329,0.400,0.462,0.286


In [19]:
# Show pitch-mix percentages for high-volume pitchers.
pitch_mix_sql = f"""
WITH agg AS (
  SELECT
    pitcher_id,
    ANY_VALUE(pitcher_name) AS pitcher_name,
    SUM(total_pitches) AS total_pitches,
    AVG(pct_four_seam) AS pct_four_seam,
    AVG(pct_sinker) AS pct_sinker,
    AVG(pct_cutter) AS pct_cutter,
    AVG(pct_slider) AS pct_slider,
    AVG(pct_sweeper) AS pct_sweeper,
    AVG(pct_curveball) AS pct_curveball,
    AVG(pct_changeup) AS pct_changeup,
    AVG(pct_splitter) AS pct_splitter,
    AVG(pct_other) AS pct_other
  FROM `{PROJECT_ID}.baseball_analytics.mart_pitcher_game_stats`
  GROUP BY pitcher_id
)
SELECT
  pitcher_name,
  total_pitches,
  pct_four_seam,
  pct_sinker,
  pct_cutter,
  pct_slider,
  pct_sweeper,
  pct_curveball,
  pct_changeup,
  pct_splitter,
  pct_other
FROM agg
WHERE total_pitches >= 50
ORDER BY total_pitches DESC
LIMIT 20
"""

pitch_mix_df = run_query(pitch_mix_sql)
pitch_mix_df


,pitcher_name,total_pitches,pct_four_seam,pct_sinker,pct_cutter,pct_slider,pct_sweeper,pct_curveball,pct_changeup,pct_splitter,pct_other
0,"Luzardo, Jesús",199,0.363,0.070,0.000,0.000,0.281,0.000,0.285,0.000,0.000
1,"Wacha, Michael",198,0.277,0.166,0.114,0.097,0.000,0.157,0.188,0.000,0.000
2,"Ryan, Joe",197,0.431,0.091,0.000,0.025,0.218,0.000,0.000,0.041,0.193
3,"Bradish, Kyle",194,0.258,0.294,0.000,0.218,0.000,0.230,0.000,0.000,0.000
4,"Yesavage, Trey",193,0.404,0.000,0.000,0.269,0.000,0.000,0.000,0.326,0.000
5,"Ashcraft, Braxton",193,0.290,0.098,0.000,0.379,0.000,0.181,0.000,0.052,0.000
6,"Baz, Shane",193,0.341,0.062,0.167,0.000,0.000,0.000,0.072,0.000,0.357
7,"Rodriguez, Eduardo",193,0.511,0.047,0.067,0.000,0.000,0.142,0.233,0.000,0.000
8,"Nelson, Ryne",191,0.556,0.037,0.130,0.208,0.000,0.070,0.000,0.000,0.000
9,"Alcantara, Sandy",189,0.227,0.201,0.000,0.370,0.063,0.000,0.138,0.000,0.000


## Player Lookup - Mapping IDs to Names


In [17]:
# Build a player lookup table from unique batter and pitcher MLBAM IDs.
batter_ids_sql = f"""
SELECT DISTINCT CAST(batter_id AS INT64) AS player_id
FROM `{PROJECT_ID}.baseball_analytics.mart_batter_game_stats`
WHERE batter_id IS NOT NULL
"""

pitcher_ids_sql = f"""
SELECT DISTINCT CAST(pitcher_id AS INT64) AS player_id
FROM `{PROJECT_ID}.baseball_analytics.mart_pitcher_game_stats`
WHERE pitcher_id IS NOT NULL
"""

batter_ids = run_query(batter_ids_sql)["player_id"].dropna().astype(int).tolist()
pitcher_ids = run_query(pitcher_ids_sql)["player_id"].dropna().astype(int).tolist()
all_player_ids = sorted(set(batter_ids + pitcher_ids))

if all_player_ids:
    player_lookup = pybaseball.playerid_reverse_lookup(all_player_ids, key_type="mlbam")
else:
    player_lookup = pd.DataFrame()

print(f"player_lookup shape: {player_lookup.shape}")
display(player_lookup.head(10))
print("player_lookup columns:")
print(player_lookup.columns.tolist())


Gathering player lookup table. This may take a moment.
player_lookup shape: (797, 8)


,name_last,name_first,key_mlbam,key_retro,key_bbref,key_fangraphs,mlb_played_first,mlb_played_last
0,schneider,davis,676914,schnd001,schneda03,23565,2023.000,2026.000
1,durbin,caleb,702332,durbc002,durbica01,29646,2025.000,2026.000
2,lowe,brandon,664040,loweb001,lowebr01,18882,2018.000,2026.000
3,junis,jakob,596001,junij001,junisja01,13619,2017.000,2026.000
4,goodman,hunter,696100,goodh001,goodmhu01,29715,2023.000,2026.000
5,giolito,lucas,608337,gioll001,giolilu01,15474,2016.000,2025.000
6,chourio,jackson,694192,chouj001,chourja01,28806,2024.000,2025.000
7,bradish,kyle,680694,bradk001,bradiky01,24586,2022.000,2026.000
8,witt,bobby,677951,wittb002,wittbo02,25764,2022.000,2026.000
9,martin,trevor,694680,NaN,martitr01,-1,2026.000,2026.000


player_lookup columns:
['name_last', 'name_first', 'key_mlbam', 'key_retro', 'key_bbref', 'key_fangraphs', 'mlb_played_first', 'mlb_played_last']


In [ ]:
# Merge player names onto top batter results and show ranked output.
lookup_id_col = "key_mlbam" if "key_mlbam" in player_lookup.columns else "player_id"

batter_lookup_df = player_lookup.copy()
if lookup_id_col in batter_lookup_df.columns:
    batter_lookup_df[lookup_id_col] = pd.to_numeric(batter_lookup_df[lookup_id_col], errors="coerce")

top_batters_named_df = top_batters_df.copy()
top_batters_named_df["batter_id"] = pd.to_numeric(top_batters_named_df["batter_id"], errors="coerce")

top_batters_named_df = top_batters_named_df.merge(
    batter_lookup_df,
    how="left",
    left_on="batter_id",
    right_on=lookup_id_col
)

top_batters_named_df = top_batters_named_df[[
    "name_first",
    "name_last",
    "batter_id",
    "games",
    "avg_exit_velocity_mph",
    "avg_xwoba",
    "avg_hard_hit_rate",
    "total_home_runs"
]].sort_values("avg_exit_velocity_mph", ascending=False)

top_batters_named_df


In [ ]:
# Merge player names onto top pitcher results and show ranked output.
lookup_id_col = "key_mlbam" if "key_mlbam" in player_lookup.columns else "player_id"

pitcher_lookup_df = player_lookup.copy()
if lookup_id_col in pitcher_lookup_df.columns:
    pitcher_lookup_df[lookup_id_col] = pd.to_numeric(pitcher_lookup_df[lookup_id_col], errors="coerce")

top_pitchers_named_df = top_pitchers_df.copy()
top_pitchers_named_df["pitcher_id"] = pd.to_numeric(top_pitchers_named_df["pitcher_id"], errors="coerce")

top_pitchers_named_df = top_pitchers_named_df.merge(
    pitcher_lookup_df,
    how="left",
    left_on="pitcher_id",
    right_on=lookup_id_col
)

top_pitchers_named_df = top_pitchers_named_df[[
    "name_first",
    "name_last",
    "pitcher_id",
    "games",
    "total_pitches",
    "avg_velocity",
    "avg_whiff_rate",
    "avg_hard_hit_allowed_rate"
]].sort_values("avg_whiff_rate", ascending=False)

top_pitchers_named_df


## Data Quality Checks


In [ ]:
# Calculate null rates for selected staging columns.
null_rate_sql = f"""
WITH total AS (
  SELECT COUNT(*) AS total_rows
  FROM `{PROJECT_ID}.baseball_analytics.stg_statcast_pitches`
)
SELECT 'exit_velocity_mph' AS column_name, COUNTIF(exit_velocity_mph IS NULL) AS null_count, t.total_rows, SAFE_DIVIDE(COUNTIF(exit_velocity_mph IS NULL), t.total_rows) AS null_pct
FROM `{PROJECT_ID}.baseball_analytics.stg_statcast_pitches` CROSS JOIN total t
GROUP BY t.total_rows
UNION ALL
SELECT 'launch_angle_deg', COUNTIF(launch_angle_deg IS NULL), t.total_rows, SAFE_DIVIDE(COUNTIF(launch_angle_deg IS NULL), t.total_rows)
FROM `{PROJECT_ID}.baseball_analytics.stg_statcast_pitches` CROSS JOIN total t
GROUP BY t.total_rows
UNION ALL
SELECT 'xba', COUNTIF(xba IS NULL), t.total_rows, SAFE_DIVIDE(COUNTIF(xba IS NULL), t.total_rows)
FROM `{PROJECT_ID}.baseball_analytics.stg_statcast_pitches` CROSS JOIN total t
GROUP BY t.total_rows
UNION ALL
SELECT 'xwoba', COUNTIF(xwoba IS NULL), t.total_rows, SAFE_DIVIDE(COUNTIF(xwoba IS NULL), t.total_rows)
FROM `{PROJECT_ID}.baseball_analytics.stg_statcast_pitches` CROSS JOIN total t
GROUP BY t.total_rows
UNION ALL
SELECT 'xslg', COUNTIF(xslg IS NULL), t.total_rows, SAFE_DIVIDE(COUNTIF(xslg IS NULL), t.total_rows)
FROM `{PROJECT_ID}.baseball_analytics.stg_statcast_pitches` CROSS JOIN total t
GROUP BY t.total_rows
UNION ALL
SELECT 'bat_speed', COUNTIF(bat_speed IS NULL), t.total_rows, SAFE_DIVIDE(COUNTIF(bat_speed IS NULL), t.total_rows)
FROM `{PROJECT_ID}.baseball_analytics.stg_statcast_pitches` CROSS JOIN total t
GROUP BY t.total_rows
UNION ALL
SELECT 'swing_length', COUNTIF(swing_length IS NULL), t.total_rows, SAFE_DIVIDE(COUNTIF(swing_length IS NULL), t.total_rows)
FROM `{PROJECT_ID}.baseball_analytics.stg_statcast_pitches` CROSS JOIN total t
GROUP BY t.total_rows
UNION ALL
SELECT 'spin_rate_rpm', COUNTIF(spin_rate_rpm IS NULL), t.total_rows, SAFE_DIVIDE(COUNTIF(spin_rate_rpm IS NULL), t.total_rows)
FROM `{PROJECT_ID}.baseball_analytics.stg_statcast_pitches` CROSS JOIN total t
GROUP BY t.total_rows
ORDER BY null_pct DESC
"""

null_rate_df = run_query(null_rate_sql)
null_rate_df


In [ ]:
# Show pitch volume by game date to verify coverage and identify potential gaps.
game_date_distribution_sql = f"""
SELECT
  game_date,
  COUNT(*) AS pitch_count
FROM `{PROJECT_ID}.baseball_analytics.stg_statcast_pitches`
GROUP BY game_date
ORDER BY game_date
"""

game_date_distribution_df = run_query(game_date_distribution_sql)
game_date_distribution_df
